In [11]:
from miditok import REMI
import symusic
import pandas as pd

from symusic import Score 

In [12]:
tokenizer = REMI()

midi_path = "test_ballade_n1.mid"
midi = Score(midi_path)

print (f"nb de pistes : {len(midi.tracks)}" )
print (f"durée de la musique :{midi.ticks_per_quarter} ticks par noire")

nb de pistes : 1
durée de la musique :192 ticks par noire


In [13]:
tokens = tokenizer(midi)

vocab_size = len(tokenizer)
print (vocab_size)

284


In [14]:
nb_tok= 30
token= tokens[0]
sample_ids= token.ids[:nb_tok]


event = [tokenizer[token_id] for token_id in sample_ids]
event

['Bar_None',
 'Position_4',
 'Pitch_48',
 'Velocity_71',
 'Duration_7.3.4',
 'Pitch_24',
 'Velocity_71',
 'Duration_10.2.4',
 'Pitch_36',
 'Velocity_71',
 'Duration_10.2.4',
 'Bar_None',
 'Position_20',
 'Pitch_39',
 'Velocity_67',
 'Duration_4.2.4',
 'Pitch_51',
 'Velocity_67',
 'Duration_4.2.4',
 'Bar_None',
 'Position_0',
 'Pitch_56',
 'Velocity_71',
 'Duration_2.5.8',
 'Position_1',
 'Pitch_44',
 'Velocity_67',
 'Duration_2.4.8',
 'Position_8',
 'Pitch_46']

In [15]:
event_type= [ev.split("_")[0] if "_" in ev else ev for ev in event]
event_value= [ev.split("_")[1] if "_" in ev else ev for ev in event]

df_tokens = pd.DataFrame({
    "Steap": range(nb_tok),
    "ID": sample_ids,
    "Event type" : event_type,
    "Event value ": event_value,
    "Raw traduction" : event

})

df_tokens

#Structure : Bar ->Position -> pitch -> velocity -> duration

,Steap,ID,Event type,Event value,Raw traduction
0,0,4,Bar,None,Bar_None
1,1,194,Position,4,Position_4
2,2,32,Pitch,48,Pitch_48
3,3,111,Velocity,71,Velocity_71
4,4,172,Duration,7.3.4,Duration_7.3.4
5,5,8,Pitch,24,Pitch_24
6,6,111,Velocity,71,Velocity_71
7,7,183,Duration,10.2.4,Duration_10.2.4
8,8,20,Pitch,36,Pitch_36
9,9,111,Velocity,71,Velocity_71


## Conversion into tensor

In [16]:
import torch
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.backends.cuda.is_available():
    device = 'cuda'
else : 
    device = 'cpu'

data = torch.tensor(token.ids, dtype = torch.long)

n = int (0.9 *len(data))
train_data = data[:n]
val_data = data[n:]

## Structure of datat : 



input : [i, i+1,i+2,..., i+block_size]  -------> output : [i+1, i+2, ....., i + block_size +1]

In [17]:
block_size = 256
batch_size = 16

def get_batch(section):
    d= train_data if section == "train" else val_data

    i = torch.randint(len(d)-block_size, (batch_size,))


    x= torch.stack([d[j:j+block_size] for j in i])

    y = torch.stack([d[j+1:j+block_size+1] for j in i])


    x= x.to(device)
    y = y.to(device)

    return x,y


b = 0
length_to_show = 5

Xb, Yb = get_batch('train')

for t in range(length_to_show):
    # Le contexte s'allonge à chaque étape
    context = Xb[b, :t+1].tolist()
    # La cible est toujours le token immédiatement après ce contexte
    target = Yb[b, t].item()
    print (f"{context} ---->{target}")

[199] ---->37
[199, 37] ---->113
[199, 37, 113] ---->131
[199, 37, 113, 131] ---->40
[199, 37, 113, 131, 40] ---->114


## Model definition

In [18]:
import torch.nn as nn
import torch.nn.functional as F


class SingleHead(nn.Module):

    def __init__(self,head_size):
            
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))


    def forward(self,x) : 

        B , T, C = x.shape #(batch , time , head_size)

        k=self.key(x)
        q = self.query(x)

        attention = q@k.transpose(-2,-1) *k.shape[-1]**0.5

        attention = attention.masked_fill(self.tril[:T,:T]==0,float ('-inf'))

        attention = F.softmax(attention,dim = -1)

        V= self.value(x)

        out = attention @V

        return out


class Block(nn.Module):

    def __init__(self,n_embd, n_head):

        super().__init__()

        head_size= n_embd//n_head

        self.sa = nn.ModuleList([SingleHead(head_size) for _ in range(n_head)])

        self.proj = nn.Linear (n_embd,n_embd)

        self.ffw = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd)
        )

        self.ln1 = nn.LayerNorm(n_embd)

        self.ln2= nn.LayerNorm(n_embd)

    def forward (self,x):


        sa_out = torch.cat([h(self.ln1(x)) for h in self.sa],dim = -1)


        x=x+self.proj(sa_out)
        x=x+self.ffw(self.ln2(x))

        return x




class MusicGEN(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim=n_embd)

        self.positional = nn.Embedding(block_size,n_embd)


        self.blocks = nn.Sequential(*[Block (n_embd, n_head) for _ in range (n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)

        self.final=nn.Linear(n_embd,vocab_size)


    def forward (self, idx, target =None):


        B,T = idx.shape

        tok_embd = self.embedding(idx) #(B,T,n_embd)
        pos_embd = self.positional(torch.arange(T,device = device)) #(T,n_embd)

        x= tok_embd+pos_embd

        x=self.blocks(x)
        x=self.ln_f(x)

        logits = self.final(x) 


        if target is None : 
            loss = None

        else : 

            logits = logits.view(B*T,vocab_size)

            target = target.view(B*T)

            loss = F.cross_entropy(logits,target)

        return logits,loss






In [19]:


vocab_size = 284 
block_size = 256 # Longueur du contexte musical vu en une fois
n_embd = 128     # Taille de la représentation vectorielle de chaque token
n_head = 4       # Nombre de "têtes" d'attention
n_layers = 4      # Nombre de couches empilées (profondeur du réseau)

model = MusicGEN().to(device)

print(f"Nombre total de paramètres à entraîner : {sum(p.numel() for p in model.parameters()) / 1e6:.2f} Millions")

# TEST : On fait passer notre Batch d'essai (Xb, Yb de l'étape précédente) dans le modèle non-entraîné
logits, loss = model(Xb, Yb)

print(f"\nDimensions de sortie (Logits) : {logits.shape}")
print(f"Erreur initiale (Loss) : {loss.item():.4f}")

Nombre total de paramètres à entraîner : 0.90 Millions

Dimensions de sortie (Logits) : torch.Size([4096, 284])
Erreur initiale (Loss) : 5.7938


On part bien de quelque chose de completement aléatoir avec une loss proche de -ln(1/284) = 5.6

# Boucle d'entrainement 

In [20]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_steps = 1000
eval_interval = 200

model.train() #mode apprentissage


for step in range(max_steps):
    xb, yb = get_batch('train')

    logits,loss = model(xb,yb)

    optimizer.zero_grad() 
    loss.backward()

    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        print(f"Étape {step} | Loss d'entraînement : {loss.item():.4f}")


Étape 0 | Loss d'entraînement : 5.8031
Étape 200 | Loss d'entraînement : 3.1501
Étape 400 | Loss d'entraînement : 3.0001
Étape 600 | Loss d'entraînement : 2.8099
Étape 800 | Loss d'entraînement : 2.6885
Étape 999 | Loss d'entraînement : 2.5065


# Generation

In [37]:
@torch.no_grad()

def generate_music(model,context,max_new_tokens):

    model.eval()

    for _ in range(max_new_tokens):

        context_cropped = context[:,-block_size:] # On ne garde que le 256 dernières notes car on un un block size de 256


        logits,_ = model(context_cropped)

        logits = logits[:,-1,:] # dernier token prédit

        #Conversion en probabilités
        probs = F.softmax(logits,dim=1)

        next_note = torch.multinomial(probs,num_samples = 1) # On choisit la prochaine note selon la distribution obtenue

        context = torch.cat((context,next_note),dim = 1)

    return context

In [43]:
amorce_silence = ['Bar_None', "Position_16"]

token_ids = [tokenizer[t] for t in amorce_silence]

context = torch.tensor([token_ids],dtype = torch.long,device = device)


musique_generee_ids = generate_music(model, context, max_new_tokens=500)

ids_list = musique_generee_ids[0].tolist() #on a un batch de taille 1 donc on récupère juste musique_generee_ids[0] que l'on met sous forme de liste


# Retour au format midi et export

from miditok import TokSequence

seq_genere = TokSequence(ids=ids_list) # Creation d'un obejt TokSequence avec notre liste
tokenizer.complete_sequence(seq_genere)
score_genere = tokenizer.decode([seq_genere]) #conversion en objet symusic.Score (comme au début)

fichier_sortie = "output/musique.mid"
score_genere.dump_midi(fichier_sortie)

